# #1 Import libraries

In [0]:
import pandas as pd
import numpy as np
from datetime import datetime
import pyarrow as pa
import pyarrow.parquet as pq

# #2 Import dataset

In [0]:
df_raw = pd.read_csv("../../data/raw/public_events_toronto.csv")
df_raw.head()


In [0]:
spark_df = spark.createDataFrame(df_raw)
BRONZE_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/bronze/public_events"

spark_df.write.mode("overwrite").parquet(BRONZE_DIR)

# #3 Data info & Data preprocessing



3.1 data info

In [0]:
df_raw.info()

In [0]:
df = df_raw.copy()
df['Event Category'].unique()

In [0]:
print("START TIMES", df['Event Time Starts'].unique())
print("\nEND TIMES:" , df['Event Time Ends'].unique())

3.2 data normalization

In [0]:
# Normaliztion of column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [0]:
# Normalization of event categories
df["event_category"] = df["event_category"].str.split("/").str[0].str.strip()
df['event_category'].unique()

In [0]:
# Transformation of timme
df['event_start_date'] = pd.to_datetime(df['event_start_date'], errors='coerce')
df['event_end_date'] = pd.to_datetime(df['event_end_date'], errors='coerce')

# Normalization of time
df["event_time_starts"] = pd.to_datetime(df["event_time_starts"], errors="coerce").dt.round("H").dt.strftime("%H:00")
df["event_time_ends"] = pd.to_datetime(df["event_time_ends"], errors="coerce").dt.round("H").dt.strftime("%H:00")



In [0]:
df.info()

3.3 data cleaning

In [0]:
df = df.drop_duplicates()

In [0]:
# Filtering by interested dates (OCT-2022 to SEP-2024)

start_range = pd.Timestamp("2022-10-01")
end_range = pd.Timestamp("2024-09-30")

df = df[
    (df["event_start_date"] >= start_range) &
    (df["event_start_date"] <= end_range)
]

3.4 Feature extraction

In [0]:
df["event_id"] = df.index.astype(str)
df = df[["event_id"] + [col for col in df.columns if col != "event_id"]]

# Creation of new columns (YEAR, MONTH and DAY) based on event dates
df["event_start_date_y"] = df["event_start_date"].dt.year
df["event_start_date_m"] = df["event_start_date"].dt.month
df["event_start_date_d"] = df["event_start_date"].dt.day

df["event_end_date_y"] = df["event_end_date"].dt.year
df["event_end_date_m"] = df["event_end_date"].dt.month
df["event_end_date_d"] = df["event_end_date"].dt.day

In [0]:
df.head(10)

# #4. Data written in dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver/public_events

In [0]:
#save public events cleaned as csv in proccessed folder
df.to_csv("../../data/processed/public_events_cleaned.csv", index=False,encoding="utf-8")

In [0]:
spark_df_cleaned = spark.createDataFrame(df)
SILVER_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver/public_events"

spark_df.write.mode("overwrite").parquet(SILVER_DIR)